# Day 076 — Exercise 3: find_elements and answer_about_screen

**What you'll build:** Two more vision tools that complete the perception layer.

**Why it matters:** `find_elements` is the most useful tool for UI understanding: ask for all buttons, menus, or input fields by type. `answer_about_screen` is the pass-through for ad-hoc visual questions.

In [ ]:
from PIL import Image as _PILImage

def _make_mock_image(width=100, height=100, color=(100, 100, 100)):
    return _PILImage.new('RGB', (width, height), color=color)
_mock_analyze_fn    = lambda img, q: 'MOCK:' + q[:16]
import io, base64

def analyze_screenshot(image, question, analyze_fn=None):
    if analyze_fn is not None:
        return analyze_fn(image, question)
    import ollama
    buf = io.BytesIO()
    image.save(buf, format='PNG')
    img_b64 = base64.b64encode(buf.getvalue()).decode()
    resp = ollama.chat(
        model='llava',
        messages=[{'role': 'user', 'content': question, 'images': [img_b64]}],
    )
    return resp['message']['content']

def describe_screen(image, analyze_fn=None):
    return analyze_screenshot(
        image, 'Describe what you see on this screen in detail.',
        analyze_fn=analyze_fn)

def read_screen_text(image, analyze_fn=None):
    return analyze_screenshot(
        image, 'Extract all visible text from this image exactly as it appears.',
        analyze_fn=analyze_fn)

def find_elements(image, element_type, analyze_fn=None):
    question = (f'List all {element_type} elements visible in this screenshot. '
                'Be specific about their labels, text, or content.')
    return analyze_screenshot(image, question, analyze_fn=analyze_fn)

def answer_about_screen(image, question, analyze_fn=None):
    return analyze_screenshot(image, question, analyze_fn=analyze_fn)


## Task

1. `find_elements(image, element_type, analyze_fn=None) -> str`
   - Build: `question = f'List all {element_type} elements visible in this screenshot. Be specific about their labels, text, or content.'`
   - Return `analyze_screenshot(image, question, analyze_fn=analyze_fn)`

2. `answer_about_screen(image, question, analyze_fn=None) -> str`
   - One line: pass `question` straight through to `analyze_screenshot`

## Your Implementation

In [ ]:
def find_elements(image, element_type, analyze_fn=None):
    """Find UI elements of a given type on screen."""
    raise NotImplementedError

def answer_about_screen(image, question, analyze_fn=None):
    """Answer an ad-hoc question about what is visible on screen."""
    raise NotImplementedError


In [ ]:
def find_elements(image, element_type, analyze_fn=None):
    question = (
        f'List all {element_type} elements visible in this screenshot. '
        'Be specific about their labels, text, or content.'
    )
    return analyze_screenshot(image, question, analyze_fn=analyze_fn)

def answer_about_screen(image, question, analyze_fn=None):
    return analyze_screenshot(image, question, analyze_fn=analyze_fn)


## Automated checks

In [ ]:

score, total = 0, 4
try:
    from PIL import Image as PILImage
    img = PILImage.new('RGB', (100, 100))

    questions = []
    find_elements(img, 'button',
                  analyze_fn=lambda i, q: (questions.append(q), 'FOUND')[1])
    assert questions and 'button' in questions[0], f"element_type not in question: {questions}"
    score += 1; print("✅ find_elements includes element_type in question")

    r = find_elements(img, 'menu', analyze_fn=_mock_analyze_fn)
    assert isinstance(r, str)
    score += 1; print("✅ find_elements returns str")

    qs = []
    answer_about_screen(img, 'How many windows?',
                        analyze_fn=lambda i, q: (qs.append(q), 'A')[1])
    assert qs and qs[0] == 'How many windows?', f"question not passed: {qs}"
    score += 1; print("✅ answer_about_screen passes question to analyze_fn")

    a = answer_about_screen(img, 'Q?', analyze_fn=_mock_analyze_fn)
    assert isinstance(a, str)
    score += 1; print("✅ answer_about_screen returns str")

except Exception as e:
    print(f"❌ {e}")

print(f"\n{score}/{total} checks passed")
if score == total:
    print("\U0001f389 Exercise complete!")


## Solution

<details><summary>Reveal</summary>

```python
def find_elements(image, element_type, analyze_fn=None):
    question = (
        f'List all {element_type} elements visible in this screenshot. '
        'Be specific about their labels, text, or content.'
    )
    return analyze_screenshot(image, question, analyze_fn=analyze_fn)

def answer_about_screen(image, question, analyze_fn=None):
    return analyze_screenshot(image, question, analyze_fn=analyze_fn)
```

**Why is element_type in the question?** The vision model has no other way to know what you are looking for. Embedding it in the question text is the zero-shot prompt engineering pattern.

</details>